# 07 — Pre-`fMRIPrep` Scanner Parameter Quality Control & Exclusions

## 1. Overview & Scientific Rationale

In **Step 5** (*Post-Clinica Metadata Correction and Data Quality Checks*, Table 2, p. 6) of Rutherford et al., *"A Pipeline for ADNI Resting-State Functional MRI Processing and Quality Control"* (`data/A Pipeline for ADNI Resting-State Functional MRI.pdf`), the authors evaluate five scanner-acquisition parameter heuristics prior to running `fMRIPrep`:

| Heuristic Metric | Paper's Table 2 Exclusion Rule (ADNI GO/2/3 only) | Rationale in Paper |
| :--- | :--- | :--- |
| **1. Repetition Time (`TR`)** | Outside `[0.5–1.0]` s or `[2.9–3.1]` s | Detects mislabeled scans or protocol inconsistencies |
| **2. Scan Depth (`dim3 × pixdim3`)** | Outside `[155, 180]` mm | Ensures adequate superior–inferior brain coverage |
| **3. Minimum Total Scan Duration** | `< 300` seconds (`nt × TR < 300 s`) | Ensures $\ge 5$ minutes of resting-state fMRI data |
| **4. `PercentPhaseFOV`** | `< 72` | Indicates spatial undersampling along phase-encoding axis |
| **5. Receive Coil (`CoilString`)** | `Q-Body` or `BODY` coils | Non-standard body coils with severely degraded SNR |

---

### Why the Paper's Raw Thresholds Must Be Adapted for Our Dataset

Because Rutherford et al. used Clinica, their pipeline **excluded all `ADNI4` scans**. In contrast, our custom BIDS dataset includes `ADNI4` sessions, which introduced a new official Philips/Siemens/GE multiband protocol (`TR = 1.50 s`, `51` slices $\times 3.0\text{ mm} = 153.0\text{ mm}$ scan depth, `197` volumes $\times 1.50\text{ s} = 295.5\text{ s}$ duration). Additionally, 12 single-band ADNI scans have `97–98` volumes $\times 3.0\text{ s} = 291–294\text{ s}$ (`4.85–4.90` minutes) because 2–3 initial dummy equilibration volumes were discarded automatically by the scanner.

Applying the paper's unadjusted cutoffs would erroneously discard `175` valid `ADNI4` scans (`TR = 1.50 s`, `153.0 mm` depth, `295.5 s` duration) and `12` valid `291–294 s` scans. Instead, we target the **4 unambiguous categories of acquisition/protocol errors (`49` sessions in total across our `3,267` BIDS sessions)**:

---

### Detailed Explanation of the 4 Exclusion Criteria (`49` Excluded Sessions)

#### Criterion 1: Ultra-Slow / Misconfigured Repetition Time (`TR >= 5.0 s`) — **`28` Sessions Excluded**
* **What we found:** `28` Philips scans (`27` in `ADNIGO/ADNI2`, `1` in `ADNI3`) were acquired with `TR ≈ 6.02 s` (`6.015 s` to `6.024 s`), twice the standard `3.0 s` ADNI single-band protocol.
* **Why they are excluded:**
  1. **Nyquist Frequency Violation & Physiological Aliasing:** By the Nyquist–Shannon sampling theorem, a sampling interval of $\text{TR} \approx 6.02\text{ s}$ yields a Nyquist frequency of:
     $$f_{\text{Nyquist}} = \frac{1}{2 \cdot \text{TR}} \approx \frac{1}{12.04\text{ s}} \approx 0.083\text{ Hz}$$
     This barely reaches the upper edge of the standard resting-state BOLD passband ($0.01\text{–}0.08\text{ Hz}$ or $0.009\text{–}0.08\text{ Hz}$), causing severe aliasing of cardiac ($\sim 1.0\text{ Hz}$) and respiratory ($\sim 0.25\text{ Hz}$) cycles into the resting-state frequency band.
  2. **Invalid Slice-Timing Interpolation:** Interpolating interleaved slices acquired up to `6.0 seconds` apart violates the temporal smoothness assumptions of sinc/Fourier slice-timing correction (`fMRIPrep` / AFNI `3dTshift`), where the hemodynamic response function (HRF) peaks in $\sim 5\text{–}6\text{ s}$.

#### Criterion 2: Severe Partial-Brain Coverage (`ScanDepth < 140 mm` / `36` Axial Slices, with Non-Standard `TR = 2.0–2.25 s`) — **`15` Sessions Excluded**
* **What we found:** `14` Philips scans (`36` slices $\times 3.3125\text{ mm} = 119.25\text{–}119.27\text{ mm}$ depth, `TR = 2.25 s`) and `1` Siemens scan (`sub-023S4502/ses-20140423`, `36` slices $\times 3.50\text{ mm} = 126.00\text{ mm}$ depth, `TR = 2.00 s`, series description `epi_2s_resting_state`).
* **Why they are excluded:**
  1. **Missing Cortical & Cerebellar ROIs in 2D Connectivity Matrices:** Standard adult superior-to-inferior brain height is $\sim 145\text{–}165\text{ mm}$ (`48` slices $\approx 158.4\text{–}163.2\text{ mm}$ in ADNI GO/2/3; `51` slices $\times 3.0\text{ mm} = 153.0\text{ mm}$ in ADNI 4). A `36`-slice FOV (`119.25–126.00 mm`) cuts off **`30–40 mm` (`3–4 cm`)** of the top or bottom of the brain (superior frontoparietal cortex and/or inferior temporal/cerebellar cortex). When extracting regional time series across a whole-brain parcellation atlas to build 2D functional connectivity matrices, nodes in the truncated regions contain zeros or out-of-brain background noise, corrupting graph neural network (GNN) node features and edge weights.
  2. **Simultaneous Protocol Deviation in `TR`:** All `15` scans also deviated from the standard `3.0 s` ADNI single-band `TR` (`14` at `2.25 s`, `1` at `2.00 s`) because the scanner operator reduced slice count from `48` to `36` and shortened `TR` proportionally ($36 / 48 \times 3.0\text{ s} = 2.25\text{ s}$).

#### Criterion 3: Aborted / Severely Truncated Acquisitions (`Duration < 250 s` / `9–72` Volumes) — **`4` Sessions Excluded**
* **What we found:** Four Siemens ADNI3 sessions terminated prematurely mid-scan:
  * `sub-007S4272/ses-20230117` (`I1663969`): `9` volumes $\times 0.607\text{ s} = 5.46\text{ s}$ (expected `976` volumes)
  * `sub-116S6100/ses-20200129` (`I1281553`): `46` volumes $\times 3.00\text{ s} = 138.0\text{ s}$ (`2.30` min; expected `197` volumes)
  * `sub-020S6901/ses-20220314` (`I1557905`): `66` volumes $\times 3.00\text{ s} = 198.0\text{ s}$ (`3.30` min; expected `197` volumes)
  * `sub-082S6629/ses-20200205` (`I1516241`): `72` volumes $\times 3.00\text{ s} = 216.0\text{ s}$ (`3.60` min; expected `197` volumes)
* **Why they are excluded:** Stable resting-state functional correlation estimation requires $\sim 5\text{ minutes}$ ($\ge 290\text{ s}$) of BOLD data. Aborted runs lasting `5.5 seconds` to `3.6 minutes` have insufficient degrees of freedom (especially after discarding initial non-steady-state frames and high-motion volumes) and yield noisy, high-variance connectivity matrices.

#### Criterion 4: Non-Standard Body Receive Coil (`CoilString` $\in \{\text{"BODY"}, \text{"Q-Body"}\}$) — **`2` Sessions Excluded**
* **What we found:**
  * `sub-098S4003/ses-20200220` (`I1293825`, GE): `CoilString = "BODY"`
  * `sub-131S5148/ses-20130418` (`I367728`, Philips): `CoilString = "Q-Body"`
* **Why they are excluded:** Instead of a multi-channel phased-array head coil placed closely around the cranium, these two BOLD runs were acquired using the scanner bore's built-in transmit/receive body coil (`BODY` / `Q-Body`). Because RF surface coil sensitivity decays rapidly with distance from the cortical surface, body-coil fMRI has drastically lower signal-to-noise ratio (SNR) and temporal SNR (tSNR), making functional connectivity estimates unreliable.

In [1]:
import gzip
import json
import shutil
import struct
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("../..").resolve()
BIDS_ROOT = PROJECT_ROOT / "data" / "raw_bids"
FINAL_MANIFEST_PATH = PROJECT_ROOT / "results" / "dataset_creation" / "final_bids_dataset_manifest.csv"
BIDS_MANIFEST_PATH = PROJECT_ROOT / "results" / "dataset_creation" / "bids_conversion_manifest.csv"
PARTICIPANTS_TSV_PATH = BIDS_ROOT / "participants.tsv"

df_final = pd.read_csv(FINAL_MANIFEST_PATH)
df_bids = pd.read_csv(BIDS_MANIFEST_PATH)

print(f"Loaded final_bids_dataset_manifest.csv: {len(df_final):,} rows")
print(f"Currently converted sessions on disk:   {(df_final['fMRI_Converted'] & df_final['T1_Converted']).sum():,}")

Loaded final_bids_dataset_manifest.csv: 3,278 rows
Currently converted sessions on disk:   3,267


## 2. Inspect NIfTI Headers & JSON Sidecars Across All `3,267` BIDS Sessions

We read the 348-byte binary NIfTI-1 header (`dim[1..4]` and `pixdim[1..4]`) and the accompanying `.json` sidecars (`func/*_task-rest_bold.json` and `anat/*_T1w.json`) for every converted session in `data/raw_bids/` to compute exact, ground-truth values of:
* `NIfTI_Slices` ($n_z = \text{dim}[3]$) and `NIfTI_Volumes` ($n_t = \text{dim}[4]$)
* `NIfTI_ScanDepth_mm` ($\text{ScanDepth} = \text{dim}[3] \times \text{pixdim}[3]$)
* `NIfTI_TR_s` ($\text{RepetitionTime}$ from `*_task-rest_bold.json`)
* `NIfTI_Duration_s` ($\text{Duration} = n_t \times \text{RepetitionTime}$)
* `PercentPhaseFOV` and `BOLD_Coil` (`CoilString` / `ReceiveCoilName`)

In [2]:
def read_nifti_dims_and_pixdims(nii_gz_path: Path):
    with gzip.open(nii_gz_path, "rb") as f:
        hdr = f.read(348)
    dims = struct.unpack("<8h", hdr[40:56])
    pixdims = struct.unpack("<8f", hdr[76:108])
    return dims[1:5], pixdims[1:5]

qc_records = []
for idx, row in df_final.iterrows():
    sub = row["BIDS_Subject"]
    ses = row["BIDS_Session"]
    bold_nii = BIDS_ROOT / sub / ses / "func" / f"{sub}_{ses}_task-rest_bold.nii.gz"
    bold_json = BIDS_ROOT / sub / ses / "func" / f"{sub}_{ses}_task-rest_bold.json"
    t1_json = BIDS_ROOT / sub / ses / "anat" / f"{sub}_{ses}_T1w.json"

    if not (row["fMRI_Converted"] and row["T1_Converted"] and bold_nii.exists()):
        qc_records.append({
            "idx": idx,
            "NIfTI_Slices": None,
            "NIfTI_Volumes": None,
            "NIfTI_TR_s": None,
            "NIfTI_ScanDepth_mm": None,
            "NIfTI_Duration_s": None,
            "BOLD_PercentPhaseFOV": None,
            "BOLD_Coil": None,
            "Rule1_SlowTR": False,
            "Rule2_PartialBrainDepth": False,
            "Rule3_AbortedDuration": False,
            "Rule4_BodyCoil": False,
            "Exclude_Pre_fMRIPrep": True,
            "Exclusion_Category": "Unsalvageable Raw DICOM (Not Converted)",
            "Exclusion_Reason": row["Comments"],
        })
        continue

    (nx, ny, nz, nt), (dx, dy, dz, dt) = read_nifti_dims_and_pixdims(bold_nii)
    with open(bold_json, "r", encoding="utf-8") as f:
        bj = json.load(f)

    tr = round(float(bj["RepetitionTime"]), 5)
    scan_depth = round(float(nz * dz), 2)
    duration = round(float(nt * tr), 2)
    pfov = bj.get("PercentPhaseFOV")
    coil_str = str(bj.get("CoilString") or bj.get("ReceiveCoilName") or row.get("ReceiveCoilName") or "")

    # Evaluate the 4 pre-fMRIPrep scanner QC exclusion criteria
    r1_slow_tr = tr >= 5.0
    r2_partial_depth = scan_depth < 140.0
    r3_aborted_dur = duration < 250.0
    r4_body_coil = coil_str.strip().upper() in {"BODY", "Q-BODY"}

    reasons = []
    categories = []
    if r1_slow_tr:
        categories.append("Criterion 1: Anomalous Slow TR (>= 5.0s)")
        reasons.append(
            f"Anomalous slow RepetitionTime (TR={tr:.3f}s, Nyquist={1/(2*tr):.3f}Hz < 0.085Hz; "
            f"causes physiological aliasing in 0.01-0.08Hz band and invalidates slice-timing interpolation)"
        )
    if r2_partial_depth:
        categories.append("Criterion 2: Partial-Brain ScanDepth (< 140mm / 36 slices)")
        reasons.append(
            f"Severe partial-brain axial coverage (ScanDepth={scan_depth:.2f}mm from {nz} slices x {dz:.4f}mm, "
            f"non-standard TR={tr:.3f}s; truncates ~3-4cm of superior/inferior cortex and atlas ROIs)"
        )
    if r3_aborted_dur:
        categories.append("Criterion 3: Aborted / Truncated Scan (< 250s)")
        reasons.append(
            f"Prematurely aborted acquisition ({nt} volumes x TR={tr:.3f}s = {duration:.2f}s total duration; "
            f"insufficient timepoints for reliable resting-state functional connectivity)"
        )
    if r4_body_coil:
        categories.append("Criterion 4: Non-Standard Body Coil (BODY / Q-Body)")
        reasons.append(
            f"Acquired with non-standard scanner body coil (CoilString='{coil_str}'); "
            f"severely degraded cortical SNR compared to phased-array head coils"
        )

    exclude = bool(r1_slow_tr or r2_partial_depth or r3_aborted_dur or r4_body_coil)
    qc_records.append({
        "idx": idx,
        "NIfTI_Slices": int(nz),
        "NIfTI_Volumes": int(nt),
        "NIfTI_TR_s": tr,
        "NIfTI_ScanDepth_mm": scan_depth,
        "NIfTI_Duration_s": duration,
        "BOLD_PercentPhaseFOV": pfov,
        "BOLD_Coil": coil_str,
        "Rule1_SlowTR": r1_slow_tr,
        "Rule2_PartialBrainDepth": r2_partial_depth,
        "Rule3_AbortedDuration": r3_aborted_dur,
        "Rule4_BodyCoil": r4_body_coil,
        "Exclude_Pre_fMRIPrep": exclude,
        "Exclusion_Category": "; ".join(categories) if exclude else "Included (Passed Pre-fMRIPrep QC)",
        "Exclusion_Reason": "; ".join(reasons) if exclude else "None",
    })

df_qc = pd.DataFrame(qc_records).set_index("idx")
for col in df_qc.columns:
    df_final[col] = df_qc[col]

df_converted = df_final[df_final["fMRI_Converted"] & df_final["T1_Converted"]]
df_excluded_49 = df_converted[df_converted["Exclude_Pre_fMRIPrep"]].copy()

print("=== Pre-fMRIPrep Scanner Parameter QC Summary (out of 3,267 converted BIDS sessions) ===")
print(df_converted["Exclusion_Category"].value_counts().to_string())
print(f"\nTotal converted sessions flagged for exclusion: {len(df_excluded_49):,}")
print(f"Total sessions passing all pre-fMRIPrep checks: {(~df_converted['Exclude_Pre_fMRIPrep']).sum():,}")

=== Pre-fMRIPrep Scanner Parameter QC Summary (out of 3,267 converted BIDS sessions) ===
Exclusion_Category
Included (Passed Pre-fMRIPrep QC)                             3218
Criterion 1: Anomalous Slow TR (>= 5.0s)                        28
Criterion 2: Partial-Brain ScanDepth (< 140mm / 36 slices)      15
Criterion 3: Aborted / Truncated Scan (< 250s)                   4
Criterion 4: Non-Standard Body Coil (BODY / Q-Body)              2

Total converted sessions flagged for exclusion: 49
Total sessions passing all pre-fMRIPrep checks: 3,218


## 3. Complete Table of the `49` Discarded Scans & Their Exclusion Reasons

Below we list all `49` sessions excluded by Criteria 1–4 alongside their exact NIfTI dimensions, `TR`, `ScanDepth`, `Duration`, `BOLD_Coil`, and `Exclusion_Reason`.

In [3]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 1000)

cols_show = [
    "BIDS_Subject", "BIDS_Session", "Image Data ID", "MRIProtocolPhase",
    "ScannerManufacturer", "NIfTI_Slices", "NIfTI_Volumes", "NIfTI_TR_s",
    "NIfTI_ScanDepth_mm", "NIfTI_Duration_s", "BOLD_Coil", "Exclusion_Category"
]
print(df_excluded_49.sort_values(["Exclusion_Category", "BIDS_Subject", "BIDS_Session"])[cols_show].to_string(index=False))

BIDS_Subject BIDS_Session Image Data ID MRIProtocolPhase ScannerManufacturer  NIfTI_Slices  NIfTI_Volumes  NIfTI_TR_s  NIfTI_ScanDepth_mm  NIfTI_Duration_s    BOLD_Coil                                         Exclusion_Category
sub-002S0295 ses-20110602       I238623     ADNIGO/ADNI2             Philips          48.0          140.0     6.02158              159.00            843.02 SENSE-Head-8                   Criterion 1: Anomalous Slow TR (>= 5.0s)
sub-002S0413 ses-20110616       I240811     ADNIGO/ADNI2             Philips          48.0          140.0     6.02158              159.00            843.02 SENSE-Head-8                   Criterion 1: Anomalous Slow TR (>= 5.0s)
sub-002S0685 ses-20110708       I243902     ADNIGO/ADNI2             Philips          48.0          140.0     6.02158              159.00            843.02 SENSE-Head-8                   Criterion 1: Anomalous Slow TR (>= 5.0s)
sub-002S0729 ses-20110816       I251176     ADNIGO/ADNI2             Philips          48

## 4. Remove the `49` Excluded Session Directories from `data/raw_bids/` & Update `participants.tsv`

To ensure `fMRIPrep` only processes the `3,218` quality-verified sessions (`3,267 - 49 = 3,218`):
1. We remove the `49` excluded session directories (`data/raw_bids/sub-*/ses-*`) from `data/raw_bids/` (raw DICOM archives remain untouched in `data/raw_fmri/` and `data/raw_smri/`).
2. If removing a session directory leaves a subject directory (`data/raw_bids/sub-*`) with zero remaining sessions, we remove the empty subject directory as well.
3. We regenerate `data/raw_bids/participants.tsv` so that it lists strictly the participants present in `data/raw_bids/`.

In [4]:
removed_sessions = 0
removed_subjects = []

for _, row in df_excluded_49.iterrows():
    sub = row["BIDS_Subject"]
    ses = row["BIDS_Session"]
    ses_dir = BIDS_ROOT / sub / ses
    if ses_dir.exists():
        shutil.rmtree(ses_dir)
        removed_sessions += 1
    sub_dir = BIDS_ROOT / sub
    if sub_dir.exists() and not any(sub_dir.iterdir()):
        sub_dir.rmdir()
        removed_subjects.append(sub)

print(f"Removed {removed_sessions} excluded session directories from data/raw_bids/")
print(f"Removed {len(removed_subjects)} empty subject directories (subjects whose only session(s) were excluded): {removed_subjects}")

# Update fMRI_Converted / BIDS_Included status in manifests
df_final["BIDS_Included"] = (df_final["fMRI_Converted"] & df_final["T1_Converted"]) & (~df_final["Exclude_Pre_fMRIPrep"])

for idx in df_excluded_49.index:
    cat = df_final.loc[idx, "Exclusion_Category"]
    reason = df_final.loc[idx, "Exclusion_Reason"]
    df_final.loc[idx, "fMRI_Converted"] = False
    df_final.loc[idx, "fMRI_Status"] = f"Excluded Pre-fMRIPrep ({cat})"
    df_final.loc[idx, "Comments"] = f"excluded pre-fMRIPrep: {reason}"

# Sync bids_conversion_manifest.csv by (BIDS_Subject, BIDS_Session, fMRI_Image_ID)
lookup_status = df_final.set_index(["BIDS_Subject", "BIDS_Session", "Image Data ID"])
for idx, row in df_bids.iterrows():
    key = (row["BIDS_Subject"], row["BIDS_Session"], row["fMRI_Image_ID"])
    if key in lookup_status.index:
        m_row = lookup_status.loc[key]
        df_bids.loc[idx, "fMRI_Converted"] = m_row["fMRI_Converted"]
        df_bids.loc[idx, "fMRI_Status"] = m_row["fMRI_Status"]
        df_bids.loc[idx, "Comments"] = m_row["Comments"]
        df_bids.loc[idx, "BIDS_Included"] = m_row["BIDS_Included"]
        df_bids.loc[idx, "Exclusion_Category"] = m_row["Exclusion_Category"]
        df_bids.loc[idx, "Exclusion_Reason"] = m_row["Exclusion_Reason"]

# Save both manifests
df_final.to_csv(FINAL_MANIFEST_PATH, index=False)
df_bids.to_csv(BIDS_MANIFEST_PATH, index=False)
print(f"Updated {FINAL_MANIFEST_PATH.name} ({df_final.shape[0]:,} rows x {df_final.shape[1]} cols)")
print(f"Updated {BIDS_MANIFEST_PATH.name} ({df_bids.shape[0]:,} rows x {df_bids.shape[1]} cols)")

# Regenerate participants.tsv for the remaining subjects in data/raw_bids/
df_included = df_final[df_final["BIDS_Included"]].copy()
df_included["Acq_Date_dt"] = pd.to_datetime(df_included["Acq Date"])
df_included = df_included.sort_values(["BIDS_Subject", "Acq_Date_dt"])

part_rows = []
for sub_id, grp in df_included.groupby("BIDS_Subject", sort=True):
    first = grp.iloc[0]
    part_rows.append({
        "participant_id": sub_id,
        "adni_ptid": first["Subject"],
        "sex": first["Sex"],
        "age_at_first_scan": first["Age"],
        "enrollment_group": first["Group"],
        "diagnosis_at_first_scan": first["Diagnosis"],
    })

df_part = pd.DataFrame(part_rows)
df_part.to_csv(PARTICIPANTS_TSV_PATH, sep="\t", index=False)
print(f"Regenerated participants.tsv with {len(df_part):,} unique participants across {len(df_included):,} BIDS sessions.")

Removed 49 excluded session directories from data/raw_bids/
Removed 7 empty subject directories (subjects whose only session(s) were excluded): ['sub-002S2043', 'sub-002S4264', 'sub-020S6901', 'sub-023S4502', 'sub-031S2017', 'sub-123S6884', 'sub-131S5148']
Updated final_bids_dataset_manifest.csv (3,278 rows x 56 cols)
Updated bids_conversion_manifest.csv (3,278 rows x 17 cols)
Regenerated participants.tsv with 1,322 unique participants across 3,218 BIDS sessions.


## 5. Final Verification of the Clean `data/raw_bids/` Dataset

We verify on disk that every remaining subject/session folder in `data/raw_bids/` matches `BIDS_Included == True` (`3,218` sessions across `1,316` unique participants), has both `func/*_bold.nii.gz` (+ `.json`) and `anat/*_T1w.nii.gz` (+ `.json`), and passes all pre-`fMRIPrep` checks.

In [5]:
sub_dirs_on_disk = sorted([p for p in BIDS_ROOT.iterdir() if p.is_dir() and p.name.startswith("sub-")])
ses_dirs_on_disk = []
incomplete_on_disk = []

for sd in sub_dirs_on_disk:
    for ses_d in sorted([p for p in sd.iterdir() if p.is_dir() and p.name.startswith("ses-")]):
        ses_dirs_on_disk.append((sd.name, ses_d.name))
        b_nii = ses_d / "func" / f"{sd.name}_{ses_d.name}_task-rest_bold.nii.gz"
        b_jsn = ses_d / "func" / f"{sd.name}_{ses_d.name}_task-rest_bold.json"
        t_nii = ses_d / "anat" / f"{sd.name}_{ses_d.name}_T1w.nii.gz"
        t_jsn = ses_d / "anat" / f"{sd.name}_{ses_d.name}_T1w.json"
        if not (b_nii.exists() and b_jsn.exists() and t_nii.exists() and t_jsn.exists()):
            incomplete_on_disk.append((sd.name, ses_d.name))

print(f"Subject directories on disk:      {len(sub_dirs_on_disk):,} (participants.tsv rows: {len(df_part):,})")
print(f"Session directories on disk:      {len(ses_dirs_on_disk):,} (manifest BIDS_Included: {int(df_final['BIDS_Included'].sum()):,})")
print(f"Incomplete session directories:   {len(incomplete_on_disk)}")
print("\n=== Final Manifest Breakdown (3,278 Total Candidate Sessions) ===")
print(df_final["Exclusion_Category"].value_counts().to_string())
print("\n=== Per-Visit Diagnosis Breakdown of the 3,218 Included BIDS Sessions ===")
print(df_included["Diagnosis"].value_counts().to_string())

assert len(sub_dirs_on_disk) == len(df_part)
assert len(ses_dirs_on_disk) == int(df_final["BIDS_Included"].sum()) == 3218
assert len(incomplete_on_disk) == 0
print("\nSUCCESS: All 3,218 BIDS sessions pass pre-fMRIPrep QC and are ready for fMRIPrep!")

Subject directories on disk:      1,322 (participants.tsv rows: 1,322)
Session directories on disk:      3,218 (manifest BIDS_Included: 3,218)
Incomplete session directories:   0

=== Final Manifest Breakdown (3,278 Total Candidate Sessions) ===
Exclusion_Category
Included (Passed Pre-fMRIPrep QC)                             3218
Criterion 1: Anomalous Slow TR (>= 5.0s)                        28
Criterion 2: Partial-Brain ScanDepth (< 140mm / 36 slices)      15
Unsalvageable Raw DICOM (Not Converted)                         11
Criterion 3: Aborted / Truncated Scan (< 250s)                   4
Criterion 4: Non-Standard Body Coil (BODY / Q-Body)              2

=== Per-Visit Diagnosis Breakdown of the 3,218 Included BIDS Sessions ===
Diagnosis
CN     1516
MCI    1286
AD      416

SUCCESS: All 3,218 BIDS sessions pass pre-fMRIPrep QC and are ready for fMRIPrep!
